In [1]:
from pathlib import Path
import pandas as pd

project_root = Path(r"C:\Users\HP-ZBOOK i7\graphrag_test")

pilot1_entities = pd.read_csv(
    project_root / "pilot_01" / "inspection" / "01_entities_all.csv"
)

pilot2_entities = pd.read_csv(
    project_root / "pilot_02" / "inspection" / "01_entities_all.csv"
)

print("Pilot 1 entities:", len(pilot1_entities))
print("Pilot 2 entities:", len(pilot2_entities))
print("Absolute change:", len(pilot2_entities) - len(pilot1_entities))

percentage_change = (
    (len(pilot2_entities) - len(pilot1_entities))
    / len(pilot1_entities)
    * 100
)

print(f"Percentage change: {percentage_change:.1f}%")

print("\nPilot 1 entity types:")
print(pilot1_entities["type"].value_counts())

print("\nPilot 2 entity types:")
print(pilot2_entities["type"].value_counts())

Pilot 1 entities: 312
Pilot 2 entities: 539
Absolute change: 227
Percentage change: 72.8%

Pilot 1 entity types:
type
ORGANIZATION    142
EVENT           121
PERSON           25
GEO              24
Name: count, dtype: int64

Pilot 2 entity types:
type
POLICY             86
TECHNOLOGY         77
TARGET             72
STAKEHOLDER        64
SUPPORT_SCHEME     58
INFRASTRUCTURE     54
CONSTRAINT         47
MARKET_METRIC      37
ORGANIZATION       31
GEOGRAPHIC_AREA    13
Name: count, dtype: int64


In [2]:
pd.set_option("display.max_colwidth", 120)

pilot2_entities[
    [
        "title",
        "type",
        "frequency",
        "degree",
        "description",
    ]
].sort_values(
    by=["degree", "frequency"],
    ascending=False,
).head(30)

,title,type,frequency,degree,description
3,PHOTOVOLTAIK,TECHNOLOGY,11,85,Photovoltaik (photovoltaics) is a renewable electricity generation technology and the core technology for decentrali...
0,ÖSTERREICH,GEOGRAPHIC_AREA,16,52,"Austria (ÖSTERREICH) is the national geographic and jurisdictional scope for all described laws, strategies, policie..."
6,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,POLICY,10,45,"The ""ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE"" (Austrian Photovoltaic Strategy) is a strategy policy for Austria that ..."
92,PV-ANLAGE,TECHNOLOGY,5,26,"PV-ANLAGE (photovoltaic system) refers to a technology for producing electricity from solar energy, encompassing ind..."
31,PHOTOVOLTAIK-STRATEGIE,POLICY,1,21,Policy type: strategy; Jurisdiction: Austria; Status: present in this document; Source meaning: the strategy defines...
347,PHOTOVOLTAIK (PV),TECHNOLOGY,2,18,Photovoltaics (PV) is a solar energy technology explicitly mentioned as central to Austria’s domestic solar market a...
211,FOTOVOLTAIKANLAGE,TECHNOLOGY,1,16,"A photovoltaic installation/system used for generating electricity from sunlight, central to the regulatory and supp..."
209,ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ELWG),POLICY,4,13,"The ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ElWG), or Electricity Industry Act, is a law in Austria that, as of early 2024, ..."
111,ENERGIEGEMEINSCHAFTEN,STAKEHOLDER,3,11,ENERGIEGEMEINSCHAFTEN refers to energy communities in Austria through which residents participate in shared photovol...
10,ERNEUERBAREN-AUSBAU-GESETZ (EAG),POLICY,5,10,"The Erneuerbaren-Ausbau-Gesetz (EAG), also known as the Renewable Expansion Act, is a law in force in Austria that s..."


In [3]:
# Select highly connected entities
top_entities = (
    pilot2_entities
    .sort_values(
        by=["degree", "frequency"],
        ascending=False,
    )
    .head(30)
)

# Select three entities from every controlled type
stratified_entities = pd.concat(
    [
        group.sample(
            n=min(3, len(group)),
            random_state=42,
        )
        for _, group in pilot2_entities.groupby("type")
    ],
    ignore_index=True,
)

# Combine and remove repeated rows
entity_audit_sample = (
    pd.concat(
        [top_entities, stratified_entities],
        ignore_index=True,
    )
    .drop_duplicates(subset="id")
    .reset_index(drop=True)
)

# Fill the sample to 60 entities if overlap reduced its size
if len(entity_audit_sample) < 60:
    remaining = pilot2_entities[
        ~pilot2_entities["id"].isin(entity_audit_sample["id"])
    ]

    additional = remaining.sample(
        n=60 - len(entity_audit_sample),
        random_state=42,
    )

    entity_audit_sample = (
        pd.concat(
            [entity_audit_sample, additional],
            ignore_index=True,
        )
        .reset_index(drop=True)
    )

print("Entity audit sample size:", len(entity_audit_sample))

print("\nEntities by type in audit sample:")
print(entity_audit_sample["type"].value_counts())

Entity audit sample size: 60

Entities by type in audit sample:
type
TECHNOLOGY         10
POLICY             10
STAKEHOLDER         9
ORGANIZATION        7
SUPPORT_SCHEME      5
GEOGRAPHIC_AREA     4
INFRASTRUCTURE      4
CONSTRAINT          4
TARGET              4
MARKET_METRIC       3
Name: count, dtype: int64


In [4]:
entity_review = entity_audit_sample[
    [
        "id",
        "title",
        "type",
        "description",
        "text_unit_ids",
        "frequency",
        "degree",
    ]
].copy()

entity_review["source_supported"] = ""
entity_review["primary_classification"] = ""
entity_review["expected_type"] = ""
entity_review["duplicate_or_alias_of"] = ""
entity_review["description_problem"] = ""
entity_review["likely_pipeline_stage"] = ""
entity_review["audit_notes"] = ""

entity_review.head(10)

,id,title,type,description,text_unit_ids,frequency,degree,source_supported,primary_classification,expected_type,duplicate_or_alias_of,description_problem,likely_pipeline_stage,audit_notes
0,8f0befe2-5c54-48a8-872e-562f55e81803,PHOTOVOLTAIK,TECHNOLOGY,Photovoltaik (photovoltaics) is a renewable electricity generation technology and the core technology for decentrali...,['d7219f76b31818b7b13bf08dda81003d752e53f7660831d5e201d105d2bc1fc19e8cc49ca6cd21e08570939d7fb3e87b45f0f5feb8575df195...,11,85,,,,,,,
1,4b33c7d9-0d87-4151-b0a4-b51575cee2d3,ÖSTERREICH,GEOGRAPHIC_AREA,"Austria (ÖSTERREICH) is the national geographic and jurisdictional scope for all described laws, strategies, policie...",['d7219f76b31818b7b13bf08dda81003d752e53f7660831d5e201d105d2bc1fc19e8cc49ca6cd21e08570939d7fb3e87b45f0f5feb8575df195...,16,52,,,,,,,
2,fac405b8-06d4-43da-b1fc-6e707fd0cade,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,POLICY,"The ""ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE"" (Austrian Photovoltaic Strategy) is a strategy policy for Austria that ...",['d7219f76b31818b7b13bf08dda81003d752e53f7660831d5e201d105d2bc1fc19e8cc49ca6cd21e08570939d7fb3e87b45f0f5feb8575df195...,10,45,,,,,,,
3,ade05329-22e9-459a-86ef-fad8af25fda2,PV-ANLAGE,TECHNOLOGY,"PV-ANLAGE (photovoltaic system) refers to a technology for producing electricity from solar energy, encompassing ind...",['e498566a78d36b3ce13304157993b9b4269045840f1d63663ab07328cc5c6a9ebc15f47c9ae340cbebf706bc8faebd35ccc4f11ae0aa4cb9a2...,5,26,,,,,,,
4,7104f7f4-94d3-4ffe-aa62-9b33ba3db307,PHOTOVOLTAIK-STRATEGIE,POLICY,Policy type: strategy; Jurisdiction: Austria; Status: present in this document; Source meaning: the strategy defines...,['50fb8f6506c2271985ee7ffe42523d2eb06abbd85be8ea5021ae2d36d260f6f63d177e59ca2da764eeaeb4a953c841282dbf214859a753f968...,1,21,,,,,,,
5,390e75de-62c8-4a9d-94f0-156f4c5963ef,PHOTOVOLTAIK (PV),TECHNOLOGY,Photovoltaics (PV) is a solar energy technology explicitly mentioned as central to Austria’s domestic solar market a...,['4d221520fe7090409cfa9cda598db30683356c340ab3584a063ffe92486f2fe7225fc056f8b997f085ff0c0f7475828879fa678b7b1709148c...,2,18,,,,,,,
6,3030e7ec-5e56-4df0-a537-ccf0b624719f,FOTOVOLTAIKANLAGE,TECHNOLOGY,"A photovoltaic installation/system used for generating electricity from sunlight, central to the regulatory and supp...",['23663988970639bea940978ccbd9150fd292856f2f31a1d587392606a64713d5dfeade66f1beea67d1cc0b0e93012337685e0eca3267f98c16...,1,16,,,,,,,
7,fbf24db5-365d-4eb4-ae70-ce610572f12b,ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ELWG),POLICY,"The ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ElWG), or Electricity Industry Act, is a law in Austria that, as of early 2024, ...",['23663988970639bea940978ccbd9150fd292856f2f31a1d587392606a64713d5dfeade66f1beea67d1cc0b0e93012337685e0eca3267f98c16...,4,13,,,,,,,
8,5973be33-cc7e-4e61-bb7a-611fb4b8f9b4,ENERGIEGEMEINSCHAFTEN,STAKEHOLDER,ENERGIEGEMEINSCHAFTEN refers to energy communities in Austria through which residents participate in shared photovol...,['e498566a78d36b3ce13304157993b9b4269045840f1d63663ab07328cc5c6a9ebc15f47c9ae340cbebf706bc8faebd35ccc4f11ae0aa4cb9a2...,3,11,,,,,,,
9,3e24e4e1-3ba2-40e9-9be3-55e4772fe719,ERNEUERBAREN-AUSBAU-GESETZ (EAG),POLICY,"The Erneuerbaren-Ausbau-Gesetz (EAG), also known as the Renewable Expansion Act, is a law in force in Austria that s...",['d7219f76b31818b7b13bf08dda81003d752e53f7660831d5e201d105d2bc1fc19e8cc49ca6cd21e08570939d7fb3e87b45f0f5feb8575df195...,5,10,,,,,,,


In [5]:
sample_path = (
    project_root
    / "pilot_02"
    / "inspection"
    / "03_entity_review_sample.csv"
)

entity_review.to_csv(
    sample_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved entity-review sample to:")
print(sample_path)

Saved entity-review sample to:
C:\Users\HP-ZBOOK i7\graphrag_test\pilot_02\inspection\03_entity_review_sample.csv


In [6]:
def audit_entity(
    title,
    source_supported,
    classification,
    expected_type,
    duplicate_of="",
    description_problem="",
    pipeline_stage="NONE",
    notes="",
):
    mask = entity_review["title"].eq(title)

    if mask.sum() == 0:
        print("Entity not found:", title)
        return

    entity_review.loc[mask, "source_supported"] = source_supported
    entity_review.loc[mask, "primary_classification"] = classification
    entity_review.loc[mask, "expected_type"] = expected_type
    entity_review.loc[mask, "duplicate_or_alias_of"] = duplicate_of
    entity_review.loc[mask, "description_problem"] = description_problem
    entity_review.loc[mask, "likely_pipeline_stage"] = pipeline_stage
    entity_review.loc[mask, "audit_notes"] = notes




In [7]:
audit_entity(
    "PHOTOVOLTAIK",
    "YES",
    "CORRECT_AND_RELEVANT",
    "TECHNOLOGY",
    notes="Photovoltaics is explicitly and repeatedly discussed. Pilot 2 correctly changed its type from EVENT to TECHNOLOGY."
)

audit_entity(
    "ÖSTERREICH",
    "YES",
    "CORRECT_AND_RELEVANT",
    "GEOGRAPHIC_AREA",
    notes="Austria is the principal geographic and policy scope of the document."
)

audit_entity(
    "ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE",
    "YES",
    "CORRECT_AND_RELEVANT",
    "POLICY",
    notes="The strategy is the source document itself. POLICY is substantially better than the Pilot 1 EVENT classification."
)

audit_entity(
    "PV-ANLAGE",
    "YES",
    "CORRECT_AND_RELEVANT",
    "TECHNOLOGY",
    description_problem="The description may combine the general technology with individual installations.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="A PV installation is relevant and is correctly no longer classified as an organization."
)

audit_entity(
    "PHOTOVOLTAIK-STRATEGIE",
    "YES",
    "DUPLICATE_OR_ALIAS",
    "POLICY",
    duplicate_of="ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE",
    pipeline_stage="ENTITY_RESOLUTION",
    notes="This is a shortened name for the Austrian Photovoltaic Strategy."
)

audit_entity(
    "PHOTOVOLTAIK (PV)",
    "YES",
    "DUPLICATE_OR_ALIAS",
    "TECHNOLOGY",
    duplicate_of="PHOTOVOLTAIK",
    pipeline_stage="ENTITY_RESOLUTION",
    notes="This is the same technology expressed with its abbreviation."
)

audit_entity(
    "FOTOVOLTAIKANLAGE",
    "YES",
    "DUPLICATE_OR_ALIAS",
    "TECHNOLOGY",
    duplicate_of="PV-ANLAGE",
    description_problem="The spelling variant should be normalized.",
    pipeline_stage="ENTITY_RESOLUTION",
    notes="This represents a photovoltaic installation and should map to the canonical PV-ANLAGE entity."
)

audit_entity(
    "ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ELWG)",
    "YES",
    "CORRECT_AND_RELEVANT",
    "POLICY",
    description_problem="The generated description should not imply legal status beyond what the document establishes.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="The proposed electricity-sector legislation is relevant to grid connection and the legal framework."
)

audit_entity(
    "ENERGIEGEMEINSCHAFTEN",
    "YES",
    "CORRECT_AND_RELEVANT",
    "STAKEHOLDER",
    description_problem="Energy communities are participation or organizational models, not individual human stakeholders.",
    pipeline_stage="SCHEMA_LIMITATION",
    notes="STAKEHOLDER is acceptable in Schema V1.1, although a future schema could use ORGANIZATIONAL_MODEL or MARKET_ACTOR."
)

audit_entity(
    "ERNEUERBAREN-AUSBAU-GESETZ (EAG)",
    "YES",
    "CORRECT_AND_RELEVANT",
    "POLICY",
    notes="The Renewable Expansion Act is explicitly cited and contains important renewable-electricity and PV objectives."
)

In [8]:
entity_review.loc[
    entity_review["primary_classification"] != "",
    [
        "title",
        "type",
        "source_supported",
        "primary_classification",
        "expected_type",
        "duplicate_or_alias_of",
        "likely_pipeline_stage",
        "audit_notes",
    ],
]

,title,type,source_supported,primary_classification,expected_type,duplicate_or_alias_of,likely_pipeline_stage,audit_notes
0,PHOTOVOLTAIK,TECHNOLOGY,YES,CORRECT_AND_RELEVANT,TECHNOLOGY,,NONE,Photovoltaics is explicitly and repeatedly discussed. Pilot 2 correctly changed its type from EVENT to TECHNOLOGY.
1,ÖSTERREICH,GEOGRAPHIC_AREA,YES,CORRECT_AND_RELEVANT,GEOGRAPHIC_AREA,,NONE,Austria is the principal geographic and policy scope of the document.
2,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,POLICY,YES,CORRECT_AND_RELEVANT,POLICY,,NONE,The strategy is the source document itself. POLICY is substantially better than the Pilot 1 EVENT classification.
3,PV-ANLAGE,TECHNOLOGY,YES,CORRECT_AND_RELEVANT,TECHNOLOGY,,DESCRIPTION_SUMMARIZATION,A PV installation is relevant and is correctly no longer classified as an organization.
4,PHOTOVOLTAIK-STRATEGIE,POLICY,YES,DUPLICATE_OR_ALIAS,POLICY,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,ENTITY_RESOLUTION,This is a shortened name for the Austrian Photovoltaic Strategy.
5,PHOTOVOLTAIK (PV),TECHNOLOGY,YES,DUPLICATE_OR_ALIAS,TECHNOLOGY,PHOTOVOLTAIK,ENTITY_RESOLUTION,This is the same technology expressed with its abbreviation.
6,FOTOVOLTAIKANLAGE,TECHNOLOGY,YES,DUPLICATE_OR_ALIAS,TECHNOLOGY,PV-ANLAGE,ENTITY_RESOLUTION,This represents a photovoltaic installation and should map to the canonical PV-ANLAGE entity.
7,ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ELWG),POLICY,YES,CORRECT_AND_RELEVANT,POLICY,,DESCRIPTION_SUMMARIZATION,The proposed electricity-sector legislation is relevant to grid connection and the legal framework.
8,ENERGIEGEMEINSCHAFTEN,STAKEHOLDER,YES,CORRECT_AND_RELEVANT,STAKEHOLDER,,SCHEMA_LIMITATION,"STAKEHOLDER is acceptable in Schema V1.1, although a future schema could use ORGANIZATIONAL_MODEL or MARKET_ACTOR."
9,ERNEUERBAREN-AUSBAU-GESETZ (EAG),POLICY,YES,CORRECT_AND_RELEVANT,POLICY,,NONE,The Renewable Expansion Act is explicitly cited and contains important renewable-electricity and PV objectives.


In [9]:
audit_entity(
    "KLIMA- UND ENERGIEFONDS",
    "YES",
    "CORRECT_AND_RELEVANT",
    "ORGANIZATION",
    notes="The Climate and Energy Fund is a named Austrian institution involved in supporting energy and PV projects."
)

audit_entity(
    "E-CONTROL",
    "YES",
    "CORRECT_AND_RELEVANT",
    "ORGANIZATION",
    notes="E-Control is the Austrian energy regulatory authority and is explicitly relevant to grid connection and market regulation."
)

audit_entity(
    "INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (NIP)",
    "YES",
    "CORRECT_AND_RELEVANT",
    "POLICY",
    description_problem="The abbreviation appears as NIP here, while the source also uses ÖNIP; these variants require normalization.",
    pipeline_stage="ENTITY_RESOLUTION",
    notes="The integrated network infrastructure plan is a relevant policy and planning document."
)

audit_entity(
    "STROMNETZ",
    "YES",
    "CORRECT_AND_RELEVANT",
    "INFRASTRUCTURE",
    notes="The electricity grid is physical and technical infrastructure required for integrating photovoltaic generation."
)

audit_entity(
    "BEVÖLKERUNG",
    "YES",
    "OVERGENERALIZED",
    "STAKEHOLDER",
    description_problem="The entire population is represented as one broad stakeholder without distinguishing citizens, consumers or affected communities.",
    pipeline_stage="ENTITY_IDENTIFICATION",
    notes="The concept is source-supported and relevant to acceptance and participation, but it is very broadly defined."
)

audit_entity(
    "PHOTOVOLTAIKANLAGEN",
    "YES",
    "DUPLICATE_OR_ALIAS",
    "TECHNOLOGY",
    duplicate_of="PV-ANLAGE",
    pipeline_stage="ENTITY_RESOLUTION",
    notes="This is the plural German form of photovoltaic installations and should be normalized to the canonical PV-ANLAGE concept."
)

audit_entity(
    "PHOTOVOLTAIK-ERZEUGUNGSSPITZEN",
    "YES",
    "CORRECT_AND_RELEVANT",
    "CONSTRAINT",
    notes="PV generation peaks are explicitly discussed as a technical grid-integration constraint."
)

audit_entity(
    "QUALIFIZIERTE FACHKRÄFTE",
    "YES",
    "CORRECT_AND_RELEVANT",
    "STAKEHOLDER",
    description_problem="The phrase denotes a workforce group rather than one organization.",
    pipeline_stage="NONE",
    notes="Qualified workers are a relevant stakeholder group because workforce availability affects PV expansion."
)

audit_entity(
    "AGRI-PV-ANLAGE",
    "YES",
    "CORRECT_AND_RELEVANT",
    "TECHNOLOGY",
    notes="Agri-PV is explicitly discussed as a photovoltaic application combining agricultural and electricity production."
)

audit_entity(
    "HEIMISCHE PV-WIRTSCHAFT",
    "YES",
    "WRONG_TYPE",
    "STAKEHOLDER",
    description_problem="The domestic PV industry is an economic sector or collective market actor, not a policy.",
    pipeline_stage="ENTITY_TYPE_SCHEMA",
    notes="The concept is relevant, but POLICY is the wrong type. Under the current schema, STAKEHOLDER is the closest available category."
)

In [11]:
reviewed_entities = entity_review[
    entity_review["primary_classification"] != ""
]

print("Reviewed entities:", len(reviewed_entities))

reviewed_entities[
    [
        "title",
        "type",
        "source_supported",
        "primary_classification",
        "expected_type",
        "duplicate_or_alias_of",
        "likely_pipeline_stage",
        "audit_notes",
    ]
]

Reviewed entities: 20


,title,type,source_supported,primary_classification,expected_type,duplicate_or_alias_of,likely_pipeline_stage,audit_notes
0,PHOTOVOLTAIK,TECHNOLOGY,YES,CORRECT_AND_RELEVANT,TECHNOLOGY,,NONE,Photovoltaics is explicitly and repeatedly discussed. Pilot 2 correctly changed its type from EVENT to TECHNOLOGY.
1,ÖSTERREICH,GEOGRAPHIC_AREA,YES,CORRECT_AND_RELEVANT,GEOGRAPHIC_AREA,,NONE,Austria is the principal geographic and policy scope of the document.
2,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,POLICY,YES,CORRECT_AND_RELEVANT,POLICY,,NONE,The strategy is the source document itself. POLICY is substantially better than the Pilot 1 EVENT classification.
3,PV-ANLAGE,TECHNOLOGY,YES,CORRECT_AND_RELEVANT,TECHNOLOGY,,DESCRIPTION_SUMMARIZATION,A PV installation is relevant and is correctly no longer classified as an organization.
4,PHOTOVOLTAIK-STRATEGIE,POLICY,YES,DUPLICATE_OR_ALIAS,POLICY,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,ENTITY_RESOLUTION,This is a shortened name for the Austrian Photovoltaic Strategy.
5,PHOTOVOLTAIK (PV),TECHNOLOGY,YES,DUPLICATE_OR_ALIAS,TECHNOLOGY,PHOTOVOLTAIK,ENTITY_RESOLUTION,This is the same technology expressed with its abbreviation.
6,FOTOVOLTAIKANLAGE,TECHNOLOGY,YES,DUPLICATE_OR_ALIAS,TECHNOLOGY,PV-ANLAGE,ENTITY_RESOLUTION,This represents a photovoltaic installation and should map to the canonical PV-ANLAGE entity.
7,ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ELWG),POLICY,YES,CORRECT_AND_RELEVANT,POLICY,,DESCRIPTION_SUMMARIZATION,The proposed electricity-sector legislation is relevant to grid connection and the legal framework.
8,ENERGIEGEMEINSCHAFTEN,STAKEHOLDER,YES,CORRECT_AND_RELEVANT,STAKEHOLDER,,SCHEMA_LIMITATION,"STAKEHOLDER is acceptable in Schema V1.1, although a future schema could use ORGANIZATIONAL_MODEL or MARKET_ACTOR."
9,ERNEUERBAREN-AUSBAU-GESETZ (EAG),POLICY,YES,CORRECT_AND_RELEVANT,POLICY,,NONE,The Renewable Expansion Act is explicitly cited and contains important renewable-electricity and PV objectives.


In [12]:
audit_entity(
    "MADE IN EUROPE-BONUS",
    "YES",
    "CORRECT_AND_RELEVANT",
    "SUPPORT_SCHEME",
    description_problem="The description must preserve that this is a proposed or described support mechanism and not imply implementation unless explicitly stated.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="The bonus is a financial or policy-support mechanism intended to encourage European PV value creation."
)

audit_entity(
    "BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE (BMK)",
    "YES",
    "CORRECT_AND_RELEVANT",
    "ORGANIZATION",
    notes="The BMK is the ministry that published the strategy and is an important institutional actor."
)

audit_entity(
    "AUSBBAUPOTENTIAL PHOTOVOLTAIK — 41 TWH BY 2040",
    "YES",
    "CORRECT_AND_RELEVANT",
    "TARGET",
    description_problem="The entity title contains a spelling error: AUSBBAUPOTENTIAL should be AUSBAUPOTENTIAL.",
    pipeline_stage="ENTITY_IDENTIFICATION",
    notes="The source supports a photovoltaic potential of 41 TWh by 2040. Pilot 2 usefully represents the value, unit and deadline as a TARGET."
)

audit_entity(
    "ENERGIEWENDE",
    "YES",
    "WRONG_TYPE",
    "PROCESS",
    description_problem="The energy transition is a broad transformation process rather than a specific policy document or legal instrument.",
    pipeline_stage="ENTITY_TYPE_SCHEMA",
    notes="The concept is relevant, but Schema V1.1 lacks a PROCESS or TRANSITION_PROCESS type."
)

audit_entity(
    "BMK-AUSBILDUNGSINITIATIVE „JUST TRANSITION“",
    "YES",
    "CORRECT_AND_RELEVANT",
    "SUPPORT_SCHEME",
    description_problem="The initiative should not be described as producing outcomes beyond those stated in the source.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="The training initiative is a relevant non-financial support measure for workforce development."
)

audit_entity(
    "PV-TECHNOLOGIEN",
    "YES",
    "OVERGENERALIZED",
    "TECHNOLOGY",
    description_problem="The entity combines multiple PV technologies into one broad concept.",
    pipeline_stage="ENTITY_IDENTIFICATION",
    notes="The concept is relevant and correctly typed, but it is broader than specific technologies or applications."
)

audit_entity(
    "ENERGIEGEMEINSCHAFTEN (EGS)",
    "YES",
    "DUPLICATE_OR_ALIAS",
    "STAKEHOLDER",
    duplicate_of="ENERGIEGEMEINSCHAFTEN",
    pipeline_stage="ENTITY_RESOLUTION",
    notes="This is the same energy-community concept with an abbreviation and should map to one canonical entity."
)

audit_entity(
    "UMWELTFÖRDERUNGSGESETZ (UFG)",
    "YES",
    "CORRECT_AND_RELEVANT",
    "POLICY",
    description_problem="The legal status and scope should remain limited to what the source explicitly states.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="The Environmental Support Act is a source-supported legal and policy instrument."
)

audit_entity(
    "BÜRGER:INNEN",
    "YES",
    "CORRECT_AND_RELEVANT",
    "STAKEHOLDER",
    description_problem="This represents a collective stakeholder group, not individual named persons.",
    pipeline_stage="NONE",
    notes="Citizens are explicitly relevant to participation, acceptance and energy communities."
)

audit_entity(
    "UNTERNEHMEN",
    "YES",
    "CORRECT_AND_RELEVANT",
    "STAKEHOLDER",
    description_problem="The entity is broad and combines companies with potentially different roles in the PV market.",
    pipeline_stage="ENTITY_IDENTIFICATION",
    notes="Companies are a relevant stakeholder group, although later analysis may distinguish manufacturers, installers, project developers and electricity companies."
)

In [13]:
reviewed_entities = entity_review[
    entity_review["primary_classification"] != ""
]

print("Reviewed entities:", len(reviewed_entities))
print()
print(reviewed_entities["primary_classification"].value_counts())

Reviewed entities: 30

primary_classification
CORRECT_AND_RELEVANT    21
DUPLICATE_OR_ALIAS       5
OVERGENERALIZED          2
WRONG_TYPE               2
Name: count, dtype: int64


In [14]:
audit_entity(
    "PROJEKTENTWICKLUNGSPHASE",
    "YES",
    "WRONG_TYPE",
    "PROCESS_STAGE",
    description_problem="A project-development phase is a stage in a process, not itself a constraint.",
    pipeline_stage="ENTITY_TYPE_SCHEMA",
    notes="The source discusses participation during project development. Schema V1.1 lacks a PROCESS_STAGE type."
)

audit_entity(
    "WASSERWIRTSCHAFT",
    "PARTIAL",
    "CORRECT_BUT_IRRELEVANT",
    "SECTOR",
    description_problem="The term mainly occurs inside the formal title or scope of the Environmental Support Act and is not developed as a PV-market factor.",
    pipeline_stage="RELEVANCE_FILTERING",
    notes="The term is present, but its independent relevance to this solar-market graph is weak."
)

audit_entity(
    "KURZ-, MITTEL- UND LANGFRISTIGER SPEICHERBEDARF",
    "YES",
    "CORRECT_AND_RELEVANT",
    "CONSTRAINT",
    notes="Different storage timescales are explicitly relevant to flexibility, grid integration and management of PV generation."
)

audit_entity(
    "EUROPÄISCHER STROMMARKT",
    "YES",
    "WRONG_TYPE",
    "MARKET",
    description_problem="An electricity market is not a geographic area, even though it has a European scope.",
    pipeline_stage="ENTITY_TYPE_SCHEMA",
    notes="The concept is relevant, but Schema V1.1 lacks a general MARKET class."
)

audit_entity(
    "BUNDESLAND",
    "YES",
    "CORRECT_AND_RELEVANT",
    "GEOGRAPHIC_AREA",
    notes="An Austrian federal state is a relevant administrative and geographic unit for spatial planning and PV implementation."
)

audit_entity(
    "ENERGIEINFRASTRUKTUREN",
    "YES",
    "CORRECT_AND_RELEVANT",
    "INFRASTRUCTURE",
    description_problem="The generated description may be broader than the exact infrastructure components stated in the supporting text.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="Energy infrastructure is directly relevant to PV integration and energy-system transformation."
)

audit_entity(
    "ÖFFENTLICHES NETZ",
    "YES",
    "CORRECT_AND_RELEVANT",
    "INFRASTRUCTURE",
    notes="The public electricity grid is explicitly discussed in relation to absorbing and distributing photovoltaic generation."
)

audit_entity(
    "BILDUNG UND AUSBILDUNG",
    "YES",
    "WRONG_TYPE",
    "SUPPORT_SCHEME",
    description_problem="Education and training are capability-building activities, not physical or digital infrastructure.",
    pipeline_stage="ENTITY_TYPE_SCHEMA",
    notes="Within the current schema, SUPPORT_SCHEME is the closest type for education and workforce-development measures."
)

audit_entity(
    "REKORDMARKEN PV-NEUINSTALLATION — 1 GW (2022); 2,5 GW (2023)",
    "YES",
    "CORRECT_AND_RELEVANT",
    "MARKET_METRIC",
    notes="This entity correctly preserves the values, units, years and Austrian context of annual PV capacity additions."
)

audit_entity(
    "RECYCLINGFÄHIGKEIT PV-PRODUKTE",
    "YES",
    "WRONG_TYPE",
    "TARGET",
    description_problem="Recyclability is presented as a desired qualitative characteristic, not as an observed numerical market metric.",
    pipeline_stage="ENTITY_TYPE_SCHEMA",
    notes="The concept is relevant to circular-economy objectives, but MARKET_METRIC is inappropriate without a measured value."
)

In [15]:
reviewed_entities = entity_review[
    entity_review["primary_classification"] != ""
]

print("Reviewed entities:", len(reviewed_entities))
print()
print(reviewed_entities["primary_classification"].value_counts())

Reviewed entities: 40

primary_classification
CORRECT_AND_RELEVANT      26
WRONG_TYPE                 6
DUPLICATE_OR_ALIAS         5
OVERGENERALIZED            2
CORRECT_BUT_IRRELEVANT     1
Name: count, dtype: int64


In [16]:
audit_entity(
    "ANTEIL FOSSILE ENERGIETRÄGER AM ENERGIEEINSATZ — KNAPP ZWEI DRITTEL (ÖSTERREICH, 2023)",
    "YES",
    "CORRECT_AND_RELEVANT",
    "MARKET_METRIC",
    description_problem="The value is expressed approximately as a fraction rather than as an exact percentage.",
    pipeline_stage="NONE",
    notes="The entity correctly retains the measured phenomenon, approximate value, geographic scope and reference year."
)

audit_entity(
    "EUROPÄISCHER RAT",
    "YES",
    "CORRECT_AND_RELEVANT",
    "ORGANIZATION",
    description_problem="The description should avoid attributing EU legislation exclusively to the European Council when other EU institutions may also be involved.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="The European Council is a source-supported EU institutional actor."
)

audit_entity(
    "EU SOLAR PV INDUSTRY ALLIANCE (ESIA)",
    "YES",
    "CORRECT_AND_RELEVANT",
    "ORGANIZATION",
    notes="The ESIA is a relevant European PV-industry alliance connected to manufacturing and the European value chain."
)

audit_entity(
    "UNI KLAGENFURT",
    "YES",
    "CORRECT_AND_RELEVANT",
    "ORGANIZATION",
    description_problem="The university's role should remain limited to the research activity explicitly mentioned in the source.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="The University of Klagenfurt is a source-supported research organization."
)

audit_entity(
    "RICHTLINIE (EU) 2018/2001",
    "YES",
    "CORRECT_AND_RELEVANT",
    "POLICY",
    notes="The EU Renewable Energy Directive is a relevant European legal and policy instrument."
)

audit_entity(
    "FTI-SCHWERPUNKT KLIMANEUTRALE STADT",
    "YES",
    "CORRECT_AND_RELEVANT",
    "POLICY",
    description_problem="The generated description should not expand the programme's PV role beyond what the document explicitly states.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="The research and innovation focus is a policy or programme relevant to climate-neutral urban energy development."
)

audit_entity(
    "AUSBILDNER:INNEN",
    "YES",
    "CORRECT_AND_RELEVANT",
    "STAKEHOLDER",
    notes="Trainers and educators are a relevant stakeholder group for addressing PV workforce and skills requirements."
)

audit_entity(
    "AKTIVE KUND:INNEN",
    "YES",
    "CORRECT_AND_RELEVANT",
    "STAKEHOLDER",
    description_problem="The description combines potentially different active-customer arrangements into one stakeholder category.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="Active customers are a source-supported market-participant and stakeholder category."
)

audit_entity(
    "LEONORE GEWESSLER",
    "YES",
    "CORRECT_AND_RELEVANT",
    "STAKEHOLDER",
    notes="Leonore Gewessler is explicitly named as the responsible federal minister and is relevant to the strategy's institutional context."
)

audit_entity(
    "NULL-PROZENT-UMSATZSTEUER FÜR PV-ANLAGEN BIS 35 KWP",
    "YES",
    "CORRECT_AND_RELEVANT",
    "SUPPORT_SCHEME",
    description_problem="Eligibility, dates and implementation status must remain exactly aligned with the source.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="This is a clearly defined financial support measure with a stated eligibility threshold."
)

In [17]:
reviewed_entities = entity_review[
    entity_review["primary_classification"] != ""
]

print("Reviewed entities:", len(reviewed_entities))
print()
print(reviewed_entities["primary_classification"].value_counts())

Reviewed entities: 50

primary_classification
CORRECT_AND_RELEVANT      36
WRONG_TYPE                 6
DUPLICATE_OR_ALIAS         5
OVERGENERALIZED            2
CORRECT_BUT_IRRELEVANT     1
Name: count, dtype: int64


In [18]:
audit_entity(
    "GENEHMIGUNGSVER­­EINFACHUNGEN UND -FREISTELLUNGEN",
    "YES",
    "CORRECT_AND_RELEVANT",
    "SUPPORT_SCHEME",
    notes="Simplified approval procedures and exemptions are non-financial policy-support mechanisms that reduce administrative barriers."
)

audit_entity(
    "FAKTENBASIERTE INFORMATION BEI PROJEKTUMSETZUNG",
    "YES",
    "CORRECT_AND_RELEVANT",
    "SUPPORT_SCHEME",
    description_problem="The description should present fact-based information as a recommended measure, not a guaranteed cause of public acceptance.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="Providing factual information is a source-supported measure for supporting acceptance during project implementation."
)

audit_entity(
    "NACHHALTIGE UND GÜNSTIGE STROMVERSORGUNG",
    "YES",
    "CORRECT_AND_RELEVANT",
    "TARGET",
    description_problem="This is a qualitative objective without a numerical value or explicit deadline.",
    pipeline_stage="NONE",
    notes="The source explicitly connects renewable-energy expansion with sustainable and affordable electricity supply."
)

audit_entity(
    "FACHKRÄFTESICHERUNG",
    "YES",
    "CORRECT_AND_RELEVANT",
    "TARGET",
    description_problem="This target is qualitative and should not be represented as already achieved.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="Securing sufficient qualified workers is a relevant PV-sector objective."
)

audit_entity(
    "RESILIENZ UND ZUVERLÄSSIGKEIT DES ENERGIESYSTEMS",
    "YES",
    "CORRECT_AND_RELEVANT",
    "TARGET",
    description_problem="The target is qualitative and does not contain a measurable value or explicit deadline.",
    pipeline_stage="NONE",
    notes="Energy-system resilience and reliability are relevant strategic objectives."
)

audit_entity(
    "FOSSILE ENERGIETRÄGER",
    "YES",
    "WRONG_TYPE",
    "ENERGY_SOURCE",
    description_problem="Fossil energy carriers are energy sources, not technologies. The statement about complete phase-out by 2040 may be stronger than the source explicitly establishes.",
    pipeline_stage="ENTITY_TYPE_SCHEMA_AND_DESCRIPTION_SUMMARIZATION",
    notes="The concept is relevant to the transition context, but Schema V1.1 lacks an ENERGY_SOURCE type."
)

audit_entity(
    "WÄRMEPUMPENSTEUERUNG",
    "YES",
    "CORRECT_AND_RELEVANT",
    "TECHNOLOGY",
    notes="Heat-pump control is a source-supported technology for flexible electricity consumption and PV integration."
)

audit_entity(
    "BAUWERKINTEGRIERTE ARCHITEKTONISCH ANGEPASSTE PHOTOVOLTAIK",
    "YES",
    "CORRECT_AND_RELEVANT",
    "TECHNOLOGY",
    notes="Building-integrated and architecturally adapted photovoltaics are a specific, source-supported PV application."
)

audit_entity(
    "BUNDESLÄNDER",
    "YES",
    "DUPLICATE_OR_ALIAS",
    "GEOGRAPHIC_AREA",
    duplicate_of="BUNDESLAND",
    description_problem="The singular and plural forms were maintained as separate graph entities.",
    pipeline_stage="ENTITY_RESOLUTION",
    notes="Both forms refer to Austrian federal states and should be normalized for controlled graph analysis."
)

audit_entity(
    "ENERGIEBERATUNGSSTELLEN DER BUNDESLÄNDER",
    "YES",
    "CORRECT_AND_RELEVANT",
    "ORGANIZATION",
    description_problem="The description should not attribute cooperation or responsibilities beyond those explicitly stated.",
    pipeline_stage="DESCRIPTION_SUMMARIZATION",
    notes="The regional energy advisory services are identifiable organizational actors supporting information and energy-community development."
)

In [19]:
reviewed_entities = entity_review[
    entity_review["primary_classification"] != ""
].copy()

print("Reviewed entities:", len(reviewed_entities))

print("\nPrimary classifications:")
print(reviewed_entities["primary_classification"].value_counts())

print("\nPrimary classification percentages:")
print(
    reviewed_entities["primary_classification"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

print("\nSource support:")
print(reviewed_entities["source_supported"].value_counts())

print("\nLikely pipeline stages:")
print(reviewed_entities["likely_pipeline_stage"].value_counts())

Reviewed entities: 60

Primary classifications:
primary_classification
CORRECT_AND_RELEVANT      44
WRONG_TYPE                 7
DUPLICATE_OR_ALIAS         6
OVERGENERALIZED            2
CORRECT_BUT_IRRELEVANT     1
Name: count, dtype: int64

Primary classification percentages:
primary_classification
CORRECT_AND_RELEVANT      73.3
WRONG_TYPE                11.7
DUPLICATE_OR_ALIAS        10.0
OVERGENERALIZED            3.3
CORRECT_BUT_IRRELEVANT     1.7
Name: proportion, dtype: float64

Source support:
source_supported
YES        59
PARTIAL     1
Name: count, dtype: int64

Likely pipeline stages:
likely_pipeline_stage
NONE                                                26
DESCRIPTION_SUMMARIZATION                           14
ENTITY_RESOLUTION                                    7
ENTITY_TYPE_SCHEMA                                   6
ENTITY_IDENTIFICATION                                4
SCHEMA_LIMITATION                                    1
RELEVANCE_FILTERING                          

In [20]:
completed_path = (
    project_root
    / "pilot_02"
    / "inspection"
    / "03_entity_review_completed.csv"
)

entity_review.to_csv(
    completed_path,
    index=False,
    encoding="utf-8-sig",
)

print("Completed entity audit saved to:")
print(completed_path)

Completed entity audit saved to:
C:\Users\HP-ZBOOK i7\graphrag_test\pilot_02\inspection\03_entity_review_completed.csv


# Pilot 2 — Entity Extraction Audit

## Audit method

A systematic sample of 60 entities was selected from the 539 entities generated in Pilot 2.

The sample combined:

- 30 highly connected entities;
- stratified entities from all ten configured entity types;
- additional randomly selected entities where required;
- evidence from the original source and the 17 GraphRAG text units.

Each entity was evaluated for source support, relevance, entity type, duplication, description quality and likely pipeline error stage.

## Results

| Classification | Count | Percentage |
|---|---:|---:|
| Correct and relevant | 44 | 73.3% |
| Wrong type | 7 | 11.7% |
| Duplicate or alias | 6 | 10.0% |
| Overgeneralized | 2 | 3.3% |
| Correct but irrelevant | 1 | 1.7% |
| **Total** | **60** | **100.0%** |

Source support was very high:

| Source support | Count | Percentage |
|---|---:|---:|
| Yes | 59 | 98.3% |
| Partial | 1 | 1.7% |

No sampled entity was classified as completely unsupported or hallucinated at the entity-identification level.

## Comparison with Pilot 1

Pilot 1 used only four generic entity types:

- ORGANIZATION
- PERSON
- GEO
- EVENT

Pilot 2 used a controlled solar-market-oriented schema containing ten entity types.

The representative Pilot 1 entity audit found:

- 13 of 57 entities correct and relevant: 22.8%;
- 28 of 57 entities wrongly typed: 49.1%;
- 8 duplicates or aliases: 14.0%.

The Pilot 2 sample found:

- 44 of 60 entities correct and relevant: 73.3%;
- 7 of 60 entities wrongly typed: 11.7%;
- 6 duplicates or aliases: 10.0%.

Therefore, the proportion of correct and relevant entities increased from approximately 22.8% to 73.3%, while wrong-type classifications decreased from approximately 49.1% to 11.7%.

Because the source document, converted text, chunk boundaries, chunk sizes and language model remained unchanged, this improvement is primarily attributable to the controlled entity schema and revised extraction prompt.

## Important improvements

Pilot 2 correctly represented several concepts that Pilot 1 had assigned to generic or incorrect types.

Examples include:

- PHOTOVOLTAIK as TECHNOLOGY rather than EVENT;
- ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE as POLICY rather than EVENT;
- PV-ANLAGE as TECHNOLOGY rather than ORGANIZATION;
- STROMNETZ as INFRASTRUCTURE;
- PV generation peaks as CONSTRAINT;
- the 41 TWh by 2040 expansion potential as TARGET;
- annual PV additions as MARKET_METRIC;
- the zero-percent VAT measure as SUPPORT_SCHEME.

The revised schema therefore produces a graph that is considerably more suitable for structured analysis of Austrian solar-market policy, technical and market factors.

## Remaining problems

### 1. Entity resolution and aliases

GraphRAG still maintains multiple entities for the same underlying concept.

Examples include:

- PHOTOVOLTAIK and PHOTOVOLTAIK (PV);
- PV-ANLAGE, FOTOVOLTAIKANLAGE and PHOTOVOLTAIKANLAGEN;
- ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE and PHOTOVOLTAIK-STRATEGIE;
- ENERGIEGEMEINSCHAFTEN and ENERGIEGEMEINSCHAFTEN (EGS);
- BUNDESLAND and BUNDESLÄNDER.

This shows that prompt-based extraction does not replace a controlled record-linkage and canonicalization stage.

### 2. Remaining schema gaps

Seven sampled entities were assigned the wrong type. Several errors occurred because the controlled schema still lacks appropriate categories.

Examples include:

- ENERGIEWENDE should be a PROCESS;
- PROJEKTENTWICKLUNGSPHASE should be a PROCESS_STAGE;
- EUROPÄISCHER STROMMARKT should be a MARKET;
- FOSSILE ENERGIETRÄGER should be an ENERGY_SOURCE;
- HEIMISCHE PV-WIRTSCHAFT is an industry or market-actor concept rather than a POLICY.

These findings do not necessarily justify immediately expanding the extraction schema. They should first be evaluated across additional documents to determine whether the missing types recur frequently.

### 3. Description summarization

Description summarization was identified as a possible issue for 14 sampled entities.

The descriptions sometimes:

- broadened an institution's role;
- implied legal or implementation status not clearly established;
- transformed recommendations into established outcomes;
- combined several arrangements into one generalized description;
- used stronger wording than the source.

Consequently, an entity may be correctly identified and typed while its generated description still requires verification.

### 4. Broad entities

Some extracted entities are source-supported but too broad for precise analysis.

Examples include:

- BEVÖLKERUNG;
- PV-TECHNOLOGIEN;
- UNTERNEHMEN.

These concepts may later require subdivision into more precise controlled entities, such as citizens, consumers, manufacturers, installers, project developers and grid operators.

## Conclusion

Pilot 2 substantially improved entity extraction quality. The controlled domain schema solved the major Pilot 1 entity-typing bottleneck and produced meaningful POLICY, TECHNOLOGY, INFRASTRUCTURE, SUPPORT_SCHEME, CONSTRAINT, MARKET_METRIC and TARGET entities.

However, the output is not yet a clean controlled knowledge graph. Entity resolution, alias normalization, description verification and several schema gaps remain.

The recommended architecture is therefore:

Source documents → GraphRAG domain extraction → evidence verification → entity canonicalization → controlled knowledge graph

GraphRAG should be retained as an extraction layer, while record linkage and final schema enforcement should be performed in a downstream controlled-KG stage.